**Chicago Ventra — PS2: Cascade / Fault-Chain Analysis**

---

Problem Statement: Analyse multi-subsystem fault chains in fare-collection devices to identify common cascade patterns, transition probabilities between subsystems, and chain clusters — enabling proactive intervention before a cascade escalates.

- Grain: (DEVICE_ID, transit_day) — 7,511 rows (all rows have has_cascade = True)
- Target/Theme: subsystem_chain (string, e.g. CHU->COMMS->OTHER) — pattern mining
- Approach:
  1. Chain frequency & length distribution
  2. Markov chain transition matrix per device type
  3. Association rules on subsystem pairs (mlxtend Apriori)
  4. TF-IDF vectorisation of chain strings → KMeans clustering
- Gold table: device_ps2_chains.parquet

Device Universe (2026-06-25): VALIDATOR ~4,200 (BMV bus only), TVM ~4,512, GATE ~2,342. Total in-scope fleet ~11,054. Oracle table count: 67 tables.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter, defaultdict
from itertools import combinations

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# mlxtend for association rules
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    from mlxtend.preprocessing import TransactionEncoder
    MLXTEND_OK = True
except ImportError:
    MLXTEND_OK = False
    print('WARNING: mlxtend not installed — association rule step will be skipped.')
    print('Install: pip install mlxtend')

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 42
print(f'All imports OK  |  mlxtend: {MLXTEND_OK}')

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────
import os
GOLD_PATH = os.environ.get(
    "GOLD_PATH",
    r'D:\Sathish\Chicago_Synthetic data\chicago_oracle_real_data\gold'
)

df = pd.read_parquet(f'{GOLD_PATH}\\device_ps2_chains.parquet')
df['transit_day'] = pd.to_datetime(df['transit_day'])
df = df.sort_values(['DEVICE_ID', 'transit_day']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["transit_day"].min().date()} → {df["transit_day"].max().date()}')
print(f'Devices: {df["DEVICE_ID"].nunique()}')
print(f'Device categories: {df["mars_device_category"].unique()}')
df.head(3)

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Nulls ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Chain Length Distribution ===')
print(df['chain_length'].describe())
print('\n=== Chain Span (minutes) ===')
print(df['chain_span_min'].describe())

In [ ]:
# Chain length and span distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['chain_length'], bins=range(1, df['chain_length'].max()+2),
             color='steelblue', edgecolor='white')
axes[0].set_title('Chain Length Distribution')
axes[0].set_xlabel('Number of events in chain')
axes[0].set_ylabel('Count')

axes[1].hist(df['chain_span_min'].clip(0, 1500), bins=40,
             color='darkorange', edgecolor='white')
axes[1].set_title('Chain Span Duration (minutes, clipped at 1500)')
axes[1].set_xlabel('Minutes')
axes[1].set_ylabel('Count')

cat_counts = df['mars_device_category'].value_counts()
axes[2].bar(cat_counts.index, cat_counts.values, color='teal')
axes[2].set_title('Fault Chains by Device Category')
axes[2].set_xlabel('Device Category')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cascade flag breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cascade_flags = {
    'has_cash_cascade': df['has_cash_cascade'].sum(),
    'has_printer_in_chain': df['has_printer_in_chain'].sum(),
    'has_gate_mech_in_chain': df['has_gate_mech_in_chain'].sum(),
    'has_csc_reader_in_chain': df['has_csc_reader_in_chain'].sum(),
    'critical_in_chain_>0': (df['critical_in_chain'] > 0).sum(),
    'oos_in_chain_>0': (df['oos_in_chain'] > 0).sum()
}
axes[0].barh(list(cascade_flags.keys()), list(cascade_flags.values()), color='salmon')
axes[0].set_title('Cascade Flag Frequency')
axes[0].set_xlabel('Chain Count')

# Critical events in chain distribution
axes[1].hist(df['critical_in_chain'], bins=range(0, df['critical_in_chain'].max()+2),
             color='crimson', edgecolor='white')
axes[1].set_title('Critical Events per Chain')
axes[1].set_xlabel('Count of critical events in chain')
axes[1].set_ylabel('Chains')

plt.tight_layout()
plt.show()

In [ ]:
# ── 1: Chain Pattern Frequency Analysis ───────────────────────────────────────
# Parse subsystem chains into lists
df['chain_list'] = df['subsystem_chain'].str.split('->')

# Top-30 unique chain patterns
chain_counter = Counter(df['subsystem_chain'])
top30_chains = chain_counter.most_common(30)
top30_df = pd.DataFrame(top30_chains, columns=['Chain', 'Count'])

plt.figure(figsize=(12, 8))
plt.barh(top30_df['Chain'][::-1], top30_df['Count'][::-1], color='royalblue')
plt.title('Top 30 Fault Chain Patterns (subsystem_chain)')
plt.xlabel('Occurrence Count')
plt.tight_layout()
plt.show()

print(f'Total unique chain patterns: {len(chain_counter)}')
print(f'Top 10 patterns:')
print(top30_df.head(10).to_string(index=False))

In [ ]:
# Subsystem frequency (individual nodes)
all_subsystems = [node for chain in df['chain_list'] for node in chain]
node_counter = Counter(all_subsystems)
node_df = pd.DataFrame(node_counter.most_common(), columns=['Subsystem', 'Count'])

plt.figure(figsize=(9, 4))
sns.barplot(data=node_df, x='Subsystem', y='Count', palette='viridis')
plt.title('Subsystem Node Frequency across All Chains')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
print(node_df.to_string(index=False))

In [ ]:
# First and Last subsystem in each chain
df['first_subsystem'] = df['chain_list'].str[0]
df['last_subsystem']  = df['chain_list'].str[-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['first_subsystem'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('First Subsystem in Chain (Chain Initiators)')
axes[0].tick_params(axis='x', rotation=30)

df['last_subsystem'].value_counts().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Last Subsystem in Chain (Chain Terminators)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── 2: Markov Chain Transition Matrix ─────────────────────────────────────────
def build_transition_matrix(chains):
    """Build a normalised Markov transition matrix from a list of chain lists."""
    transitions = defaultdict(Counter)
    for chain in chains:
        for i in range(len(chain) - 1):
            transitions[chain[i]][chain[i+1]] += 1
    # All unique states
    states = sorted(set(s for c in chains for s in c))
    mat = pd.DataFrame(0.0, index=states, columns=states)
    for src, counts in transitions.items():
        total = sum(counts.values())
        for dst, cnt in counts.items():
            mat.loc[src, dst] = cnt / total
    return mat

# Global transition matrix
global_mat = build_transition_matrix(df['chain_list'].tolist())

plt.figure(figsize=(9, 7))
sns.heatmap(global_mat, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.3, vmin=0, vmax=1)
plt.title('Markov Transition Matrix — All Devices (row→col = P(next|current))')
plt.xlabel('Next Subsystem')
plt.ylabel('Current Subsystem')
plt.tight_layout()
plt.show()

In [ ]:
# Per-device-type Markov matrices
device_cats = sorted(df['mars_device_category'].unique())
ncols = 2
nrows = (len(device_cats) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 5))
axes = axes.flatten()

for i, cat in enumerate(device_cats):
    sub_chains = df[df['mars_device_category'] == cat]['chain_list'].tolist()
    if len(sub_chains) < 5:
        axes[i].set_visible(False)
        continue
    mat = build_transition_matrix(sub_chains)
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.2, ax=axes[i], vmin=0, vmax=1,
                cbar=False)
    axes[i].set_title(f'{cat} (n={len(sub_chains)})')
    axes[i].set_xlabel('Next')
    axes[i].set_ylabel('Current')
    axes[i].tick_params(axis='x', rotation=30, labelsize=7)
    axes[i].tick_params(axis='y', labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Markov Transition Matrices by Device Category', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3: Association Rules on Subsystem Pairs ───────────────────────────────────
if MLXTEND_OK:
    # Treat each chain as a transaction of unique subsystems
    transactions = [list(set(chain)) for chain in df['chain_list']]
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    te_df = pd.DataFrame(te_array, columns=te.columns_)

    # Apriori with min support
    freq_items = apriori(te_df, min_support=0.03, use_colnames=True)
    print(f'Frequent itemsets: {len(freq_items)}')

    rules = association_rules(freq_items, metric='lift', min_threshold=1.0)
    rules = rules.sort_values('lift', ascending=False)
    print(f'Association rules (lift>=1.0): {len(rules)}')
    print('\nTop 15 Rules by Lift:')
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    print(rules[display_cols].head(15).to_string(index=False))
else:
    print('mlxtend not available — skipping association rules.')
    rules = pd.DataFrame()

In [ ]:
if MLXTEND_OK and len(rules) > 0:
    # Scatter: support vs confidence, coloured by lift
    plt.figure(figsize=(9, 5))
    sc = plt.scatter(rules['support'], rules['confidence'],
                     c=rules['lift'], cmap='plasma', alpha=0.7, s=50)
    plt.colorbar(sc, label='Lift')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.title('Association Rules: Support vs Confidence (colour=Lift)')
    plt.tight_layout()
    plt.show()

    # Top rules bar chart
    top_rules = rules.head(15).copy()
    top_rules['rule_str'] = (top_rules['antecedents'].astype(str) + ' → ' +
                              top_rules['consequents'].astype(str))
    plt.figure(figsize=(10, 6))
    plt.barh(top_rules['rule_str'][::-1], top_rules['lift'][::-1], color='mediumpurple')
    plt.xlabel('Lift')
    plt.title('Top 15 Association Rules by Lift — PS2')
    plt.tight_layout()
    plt.show()
else:
    print('No rules to plot.')

In [ ]:
# ── 4: TF-IDF + KMeans Chain Clustering ───────────────────────────────────────
# Treat each chain string as a document; subsystems separated by ' ' for TF-IDF
chain_docs = df['subsystem_chain'].str.replace('->', ' ')

tfidf = TfidfVectorizer(analyzer='word', token_pattern=r'[^\s]+',
                         ngram_range=(1, 2), min_df=2)
X_tfidf = tfidf.fit_transform(chain_docs)
X_norm  = normalize(X_tfidf)
print(f'TF-IDF matrix: {X_norm.shape}')

# Elbow method to choose K
inertias = []
K_range = range(2, 12)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_norm)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Plot for KMeans Clustering of Fault Chains')
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with chosen K=6
K_FINAL = 6
km_final = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=15)
df['cluster'] = km_final.fit_predict(X_norm)

# PCA 2D visualisation
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_norm.toarray())

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'],
                      cmap='tab10', alpha=0.5, s=15)
plt.colorbar(scatter, label='Cluster')
plt.title(f'Fault Chain Clusters (KMeans K={K_FINAL}, PCA 2D projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Cluster profiles: top chain patterns + cascade flags per cluster
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == c]
    top_chains = sub['subsystem_chain'].value_counts().head(5)
    avg_len = sub['chain_length'].mean()
    avg_crit = sub['critical_in_chain'].mean()
    cash_pct = sub['has_cash_cascade'].mean() * 100
    gate_pct = sub['has_gate_mech_in_chain'].mean() * 100
    print(f'\n── Cluster {c} (n={len(sub)}, avg_len={avg_len:.1f}, avg_crit={avg_crit:.2f},',
          f'cash_cascade={cash_pct:.0f}%, gate={gate_pct:.0f}%) ──')
    print('  Top 5 patterns:', top_chains.to_dict())

In [ ]:
# Cluster severity profile heatmap
cluster_profile = df.groupby('cluster').agg(
    avg_chain_length=('chain_length', 'mean'),
    avg_span_min=('chain_span_min', 'mean'),
    avg_critical=('critical_in_chain', 'mean'),
    avg_oos=('oos_in_chain', 'mean'),
    pct_cash=('has_cash_cascade', 'mean'),
    pct_gate=('has_gate_mech_in_chain', 'mean'),
    pct_csc=('has_csc_reader_in_chain', 'mean'),
    n=('DEVICE_ID', 'count')
).round(3)

fig, ax = plt.subplots(figsize=(11, 4))
profile_norm = (cluster_profile.drop('n', axis=1) -
                cluster_profile.drop('n', axis=1).min()) / \
               (cluster_profile.drop('n', axis=1).max() -
                cluster_profile.drop('n', axis=1).min() + 1e-9)
sns.heatmap(profile_norm.T, annot=cluster_profile.drop('n', axis=1).T,
            fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Cluster Profile Heatmap (values normalised 0→1, annotations = raw)')
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()
print(cluster_profile)

In [ ]:
# ── Device-Type Cascade Comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg chain length by device type
avg_len = df.groupby('mars_device_category')['chain_length'].mean().sort_values(ascending=False)
axes[0].bar(avg_len.index, avg_len.values, color='cadetblue')
axes[0].set_title('Average Chain Length by Device Category')
axes[0].set_ylabel('Avg Events in Chain')
axes[0].tick_params(axis='x', rotation=45)

# Cash cascade rate by device type
cash_rate = df.groupby('mars_device_category')['has_cash_cascade'].mean().sort_values(ascending=False)
axes[1].bar(cash_rate.index, cash_rate.values * 100, color='goldenrod')
axes[1].set_title('Cash Cascade Rate (%) by Device Category')
axes[1].set_ylabel('% of Chains with Cash Cascade')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Chain velocity: chains per device per day
daily_chains = df.groupby(['mars_device_category', 'transit_day']).size().reset_index(name='chain_count')
plt.figure(figsize=(12, 4))
for cat in sorted(daily_chains['mars_device_category'].unique()):
    sub = daily_chains[daily_chains['mars_device_category'] == cat]
    plt.plot(sub['transit_day'], sub['chain_count'], label=cat, alpha=0.7, linewidth=1)
plt.title('Daily Fault Chain Volume by Device Category')
plt.xlabel('Date')
plt.ylabel('Chain Count')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Key Findings — PS2 Cascade / Fault-Chain Analysis**

**Chain Patterns**
- The dataset contains **7,511 fault chains** — all rows have `has_cascade = True` (single-event entries filtered out upstream in silver)
- Chain lengths range from 2 to longer sequences; short chains (2–4 events) dominate
- `OTHER` is the most frequent subsystem node, indicating many unclassified / catch-all events
- `CHU` (Cash Handling Unit) is the most common *named* initiator of fault chains in TVM/BMV devices

**Markov Transition Highlights**
- High self-loop probability on `OTHER→OTHER` across all device types — recurring generic faults before a specific subsystem fires
- TVM devices: `CHU→COMMS` transition elevated — cash jam events often trigger communication failures
- GATE devices: `GATE_FAULT→OTHER` is the primary transition, suggesting mechanical failures cascade to generic states

**Association Rules**
- Strongest lift pairs typically involve `(CHU, BHU)` — bill and coin subsystems co-fail frequently in TVM/BMV
- `(CSC_READER, COMMS)` co-occurrence elevated — reader communication faults cluster together

**Clusters (K=6 KMeans on TF-IDF)**
- Cluster with highest `avg_critical` and `avg_span_min` represents long-duration critical cascades — highest priority for intervention
- Clusters with `cash_cascade > 50%` are TVM/BMV-dominated
- Short, single-subsystem clusters are low-severity noise chains

**Operational Recommendations**
1. **TVM/BMV cash-cascade alert:** If `CHU` appears as first event and chain length ≥ 3, escalate immediately
2. **Gate mechanical chains:** `GATE_FAULT` initiators with `chain_span_min > 120` correlate with extended OOS
3. **CSC READER + COMMS co-occurrence** warrants proactive comms-hardware inspection

**Limitations**
- `first_subsystem` / `last_subsystem` not available as pre-built columns — derived from chain string in notebook
- Synthetic data: chain patterns are algorithmically generated; real Ventra chain distributions may differ
- `OTHER` subsystem category masks specific component information